In [23]:
import pandas as pd 
pd.set_option('display.max_columns', None)
import plotly.express as px

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [24]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [25]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [28]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
import plotly.express as px



# Drop customerID as it's not useful
df = df.drop('customerID', axis=1)

print("Original Data Shape:", df.shape)
print("\nData Types:\n", df.dtypes)

# ====================== DATA CLEANING ======================
# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill any NaN in TotalCharges with 0 (safe for this dataset)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# ====================== ENCODING STRATEGY ======================

# Columns for One-Hot Encoding
onehot_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 
    'PaymentMethod'
]

# Column for Ordinal Encoding
ordinal_cols = ['Contract']

# Numerical columns
numeric_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

# ====================== COLUMN TRANSFORMER ======================

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), onehot_cols),
        ('ordinal', OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']]), ordinal_cols),
        ('numeric', 'passthrough', numeric_cols)
    ],
    remainder='drop'   # Drop Churn for now (we'll encode it separately)
)

# ====================== FIT AND TRANSFORM FEATURES ======================

X = df.drop('Churn', axis=1)
y = df['Churn']

# Transform features
X_encoded = preprocessor.fit_transform(X)

# Get feature names
onehot_feature_names = preprocessor.named_transformers_['onehot'].get_feature_names_out(onehot_cols)
feature_names = list(onehot_feature_names) + ordinal_cols + numeric_cols

# Create final encoded features DataFrame
X_final = pd.DataFrame(X_encoded, columns=feature_names)

print("\nFinal Feature Shape after Encoding:", X_final.shape)

# ====================== ENCODE TARGET (Churn) ======================

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("\nChurn Encoding Mapping:")
print(dict(zip(le.classes_, le.transform(le.classes_))))

# Combine features + target for easy EDA
df_final = X_final.copy()
df_final['Churn'] = y_encoded

# ====================== SAVE PROCESSED DATA ======================
df_final.to_csv('telco_churn_preprocessed.csv', index=False)
print("\nPreprocessed data saved as 'telco_churn_preprocessed.csv'")

# ====================== QUICK EDA ======================
# 1. Churn by Contract
fig = px.histogram(df, x='Contract', color='Churn', barmode='group',
                   title='Churn Distribution by Contract Type',
                   color_discrete_map={'Yes': 'red', 'No': 'green'})
fig.show()

# 2. Correlation Heatmap - FIXED VERSION
# Use df_final (which has only numeric values) instead of original df
fige = px.imshow(
    df_final.corr(),
    text_auto=True,
    aspect="auto",
    color_continuous_scale='RdBu_r',
    title='Correlation Heatmap of Encoded Features'
)
fige.update_layout(height=800, width=1000)
fige.show()

# Optional: Show top correlations with Churn
corr_with_churn = df_final.corr()['Churn'].sort_values(ascending=False)
print("\nTop Correlations with Churn:")
print(corr_with_churn.head(10))

Original Data Shape: (7043, 20)

Data Types:
 gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Final Feature Shape after Encoding: (7043, 29)

Churn Encoding Mapping:
{'No': 0, 'Yes': 1}

Preprocessed data saved as 'telco_churn_preprocessed.csv'



Top Correlations with Churn:
Churn                             1.000000
InternetService_Fiber optic       0.308020
PaymentMethod_Electronic check    0.301919
MonthlyCharges                    0.193356
PaperlessBilling_Yes              0.191825
SeniorCitizen                     0.150889
StreamingTV_Yes                   0.063228
StreamingMovies_Yes               0.061382
MultipleLines_Yes                 0.040102
PhoneService_Yes                  0.011942
Name: Churn, dtype: float64


In [30]:
import plotly.express as px

# Create the most compelling visualization: Churn Rate by Internet Service & Payment Method
# This combines the two strongest churn drivers

fig_story = px.bar(
    df, 
    x='InternetService', 
    color='PaymentMethod',
    barmode='group',
    facet_col='Churn',                    # Separate No / Yes churn
    title='<b>Why Customers Are Leaving: Fiber Optic + Electronic Check = Highest Churn Risk</b><br>'
          '<span style="font-size: 14px">Churn Rate is dramatically higher for Fiber Optic users, especially those paying with Electronic Check</span>',
    labels={'InternetService': 'Internet Service Type', 
            'PaymentMethod': 'Payment Method'},
    color_discrete_sequence=px.colors.qualitative.Set2,
    text_auto=True,
    height=650,
    width=1100
)

# Improve layout and annotations
fig_story.update_layout(
    showlegend=True,
    legend_title="Payment Method",
    yaxis_title="Number of Customers",
    title_x=0.5,
    font=dict(size=13)
)

# Add big annotation explaining "So What?"
fig_story.add_annotation(
    text="<b>SO WHAT?</b><br><br>"
         "• Fiber Optic has the highest churn (especially with Electronic Check)<br>"
         "• Customers expect premium service but face higher bills + payment friction<br>"
         "• Action: Improve Fiber reliability, offer discounts for longer contracts,<br>"
         "  and promote auto-pay / credit card to reduce friction",
    xref="paper", yref="paper",
    x=0.02, y=0.95,
    showarrow=False,
    align="left",
    bgcolor="rgba(255, 255, 255, 0.9)",
    bordercolor="red",
    borderwidth=2,
    borderpad=10,
    font=dict(size=14, color="black")
)

fig_story.show()

In [31]:
# Alternative: Single stacked bar with clear churn % annotation
fig_alt = px.histogram(
    df, 
    x='InternetService', 
    color='PaymentMethod',
    barmode='group',
    title='<b>The Killer Combination Driving Churn: Fiber Optic + Electronic Check</b>',
    labels={'InternetService': 'Internet Service', 'count': 'Number of Customers'},
    height=600
)

fig_alt.add_annotation(
    text="Fiber Optic + Electronic Check customers churn at the highest rate.<br>"
         "<b>Business Impact:</b> Fix service quality or pricing perception for Fiber users<br>"
         "and push them toward automatic/credit card payments + longer contracts.",
    x=1.05, y=0.85, xref="paper", yref="paper",
    showarrow=False, align="left", bgcolor="#fff2f2", bordercolor="#ff4444"
)

fig_alt.show()

In [35]:
df_final.columns

Index(['gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'Contract', 'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
       'Churn'],
      dtype='object')

In [38]:
x = df_final[['InternetService_Fiber optic', 'PaymentMethod_Electronic check', 'MonthlyCharges', 'PaperlessBilling_Yes', "SeniorCitizen"]]
y = df_final["Churn"]

In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.25, random_state=42, stratify=y_encoded)

print(f"\nTrain shape: {X_train.shape} | Test shape: {X_test.shape}")

# Fit preprocessor and transform
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

# ====================== 5. BASELINE MODEL: Logistic Regression ======================
print("\n=== Building Simple Baseline Model ===")
print("Model: Logistic Regression (5–6 key features)")

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_encoded, y_train)

# Predictions
y_pred = model.predict(X_test_encoded)
y_pred_proba = model.predict_proba(X_test_encoded)[:, 1]

# ====================== 6. EVALUATION ======================
print("\n=== Model Evaluation ===")

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# ====================== 7. SUCCESS METRIC EXPLANATION ======================
print("\n=== Success Metric: Why Precision Matters More Than Accuracy ===")
print("• In churn prediction, **Precision** is often more important than Accuracy.")
print("• High Precision means: When the model predicts 'Churn', it is usually correct.")
print("• Why? Because retaining a customer is expensive (offers, discounts, support calls).")
print("  → We want to avoid wasting resources on customers who were never going to leave (False Positives).")
print("• Accuracy can be misleading due to class imbalance (most customers do not churn).")

# ====================== 8. ETHICAL CONSIDERATION ======================
print("\n=== Ethical Consideration ===")
print("• **Avoiding bias against SeniorCitizen**")
print("  Senior citizens have higher predicted churn risk (correlation = 0.151).")
print("  → We must ensure the model does not lead to discriminatory treatment.")
print("  → Recommendation: Monitor fairness metrics (Equal Opportunity, Demographic Parity)")
print("    across SeniorCitizen groups and avoid using age-related proxies unfairly.")

# ====================== 9. FEATURE IMPORTANCE (Baseline Model) ======================
print("\n=== Baseline Model Feature Importance (Logistic Regression Coefficients) ===")

# Get feature names
onehot_names = preprocessor.named_transformers_['onehot'].get_feature_names_out(['InternetService', 'PaymentMethod', 'PaperlessBilling'])
feature_names = list(onehot_names) + ['Contract_Ordinal', 'MonthlyCharges', 'SeniorCitizen']

coefficients = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print(coefficients.round(4))

# Optional: Plot coefficients
fig = px.bar(coefficients, x='Coefficient', y='Feature', orientation='h',
             title='Feature Importance (Logistic Regression Coefficients)',
             labels={'Coefficient': 'Impact on Churn Probability'})
fig.show()





Train shape: (5282, 19) | Test shape: (1761, 19)

=== Building Simple Baseline Model ===
Model: Logistic Regression (5–6 key features)


c:\Users\Predator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning:

lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression




=== Model Evaluation ===
Accuracy : 0.8069
Precision: 0.6624
Recall   : 0.5546
F1 Score : 0.6037

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.85      0.90      0.87      1294
       Churn       0.66      0.55      0.60       467

    accuracy                           0.81      1761
   macro avg       0.76      0.73      0.74      1761
weighted avg       0.80      0.81      0.80      1761


Confusion Matrix:
[[1162  132]
 [ 208  259]]

=== Success Metric: Why Precision Matters More Than Accuracy ===
• In churn prediction, **Precision** is often more important than Accuracy.
• High Precision means: When the model predicts 'Churn', it is usually correct.
• Why? Because retaining a customer is expensive (offers, discounts, support calls).
  → We want to avoid wasting resources on customers who were never going to leave (False Positives).
• Accuracy can be misleading due to class imbalance (most customers do not churn).

=== Ethical C

ValueError: input_features is not equal to feature_names_in_